In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Imports

In [27]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
from comet_ml import Experiment
from src.costs.lse import SharedMLPLSECost
from src.models.gmm_based import GMMEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.metrics import (
    compute_BW_UVP,
    compute_metrics,
    compute_mmd,
    compute_sinkhorn_divergence,
)
from src.utils.paired import get_paired_sampler, match_gaussian_and_swiss_roll
from src.utils.train import compute_loss, update_average
from tqdm import tqdm

In [28]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [29]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [30]:
from configs.gmm_based.cost import SharedMLPLSECostConfig
from configs.gmm_based.dataset import DatasetConfig, MiniBatchConfig
from configs.gmm_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.gmm_based.train import TrainConfig

In [31]:
# Data
Q_X_UNPAIRED_SAMPLES = 16000 # 1024
R_Y_UNPAIRED_SAMPLES = 16000 # 1024
P_XY_PAIRED_SAMPLES = 16000 # 128

# Optimizer
LR_PAIRED = 3e-4
LR_UNPAIRED = 1e-3

# Sampler
PAIRED_BATCH_SIZE = 128
UNPAIRED_BATCH_SIZE = 128

G_FUNC = lambda x: x
NUM_METRIC_SAMPLES = 1024

# Train
MAX_STEPS = 25001
INIT_BY_SAMPLES = True

# Potential
Y_DIM = 2
N_POTENTIALS = 1

# Cost
M_POTENTIALS = 1
SHARED_HIDDEN_CHANNELS = [M_POTENTIALS * Y_DIM]

In [32]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
minibatch_config = MiniBatchConfig()

cost_config = SharedMLPLSECostConfig(
    m_potentials=M_POTENTIALS,
    shared_hidden_channels=SHARED_HIDDEN_CHANNELS,
)
EXP_META_INFO = (
    f"M_POTENTIALS_{M_POTENTIALS}_"
    + f"SHARED_HIDDEN_CHANNELS_{SHARED_HIDDEN_CHANNELS}_"
)

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [33]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [34]:
X_sampler = StandardNormalSampler(dim=dataset_config.x_dim, device=device)
Y_sampler = SwissRollSampler(dim=dataset_config.y_dim, device=device)

In [35]:
X_paired_train = X_sampler.sample(P_XY_PAIRED_SAMPLES)
Y_paired_train = match_gaussian_and_swiss_roll(Y_sampler, X_paired_train, 1, g_func=G_FUNC).squeeze(1)
X_paired_test = X_sampler.sample(P_XY_PAIRED_SAMPLES)
Y_paired_test = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, 1, g_func=G_FUNC).squeeze(1)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16000/16000 [00:09<00:00, 1662.50it/s]


In [36]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [37]:
X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [38]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
    usd_sampler = DatasetSampler(source_data, device=device)  # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device)  # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [39]:
cost = SharedMLPLSECost(**cost_config.model_dump())

/trinity/home/m.persiyanov/miniconda3/envs/light-gcot/lib/python3.12/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `float` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


In [40]:
model = GMMEOT(
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    cost=cost,
).to(dtype)

if INIT_BY_SAMPLES:
    model.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

In [41]:
# For EMA update
if train_config.ema_update:
    model_copy = GMMEOT(
        y_dim=Y_DIM,
        n_potentials=N_POTENTIALS,
        cost=cost,
    ).to(dtype)

## 5. Optimizers initialization

In [42]:
# unpaired_params_to_update = [model._log_w_n, model._a_n, model._log_A_n]

unpaired_params_to_update = [
    {"params": [model._log_w_n, model._a_n], "lr": opt_unpaired_config.lr},
    {"params": [model._log_A_n], "lr": opt_unpaired_config.lr * 0.1} # Slow down variance collapse
]

D_opt_unpaired = torch.optim.Adam(unpaired_params_to_update, **opt_unpaired_config.model_dump())

In [43]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump(), weight_decay=1e-4)

In [44]:
# TODO: refactor this config
EXP_NAME = (
    "GMMEOT_Swiss_Roll_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"M_{M_POTENTIALS}_"
    + f"N_{N_POTENTIALS}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    M_POTENTIALS=M_POTENTIALS,
    N_POTENTIALS=N_POTENTIALS,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [45]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(
        torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt"))
    )
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [46]:
starting_points = torch.tensor([[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]])
num_ending_points = 64

In [47]:
num_starting_points_paired = 5
indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [48]:
gt_Y_points = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, 64, g_func=G_FUNC)
gt_Y_points_for_metrics = match_gaussian_and_swiss_roll(Y_sampler, X_paired_test, NUM_METRIC_SAMPLES, g_func=G_FUNC)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16000/16000 [00:07<00:00, 2031.47it/s]


In [49]:
metrics_dict = {
    "mmd": lambda x, y: compute_mmd(x, y),
    "W_2": lambda x, y: compute_sinkhorn_divergence(x, y),
    "BW_UVP": lambda x, y: compute_BW_UVP(x, y),
}

In [50]:
experiment = Experiment(
    project_name="Light-GCOT-Swiss-Roll",
    auto_output_logging=False,
    parse_args=False,
)
experiment.set_name(EXP_NAME)
experiment.log_parameters(config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()

    # Clip gradients for both networks
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(cost.parameters(), max_norm=1.0)

    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    if step % train_config.log_every == 0:
        experiment.log_metric("Unpaired loss", D_loss_unpaired.item(), step=step)
        experiment.log_metric("Paired loss", D_loss_paired.item(), step=step)
        experiment.log_metric("Loss", D_loss, step=step)
        experiment.log_metric(
            "Train paired loss",
            compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train),
            step=step,
        )
        experiment.log_metric(
            "Test paired loss",
            compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test),
            step=step,
        )
        experiment.log_metric(
            "Test unpaired loss",
            compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test),
            step=step,
        )

        experiment.log_metric(r"$-f^c(x)$", -output_unpaired["f_c"].mean().item(), step=step)
        experiment.log_metric(r"$-f(y)$", -output_unpaired["f"].mean().item(), step=step)
        experiment.log_metric(f"lam_min(A_n)", torch.min(output_unpaired["A_n"]), step=step)
        experiment.log_metric(f"lam_max(A_n)", torch.max(output_unpaired["A_n"]), step=step)
        unconditional_metrics, conditional_metrics = compute_metrics(
            models_dict={"GMMEOT": model},
            metrics_dict=metrics_dict,
            X_sampler=X_sampler,
            Y_sampler=Y_sampler,
            starting_points=starting_points,
            gt_Y_points=gt_Y_points_for_metrics,
            num_samples=NUM_METRIC_SAMPLES,
            experiment=experiment,
        )

    if step % train_config.plot_every == 0:
        plot_A_parameters(model, experiment=experiment)
        plot_B_parameters(model.cost, starting_points, experiment=experiment)
        if num_starting_points_paired > 0:
            plot_Z_parameters(
                model, starting_points, starting_points_paired, ending_points_paired, experiment=experiment
            )
        else:
            plot_Z_parameters(model, starting_points, experiment=experiment)
        plot_swiss_roll(
            {f"P={P_XY_PAIRED_SAMPLES}, Q={Q_X_UNPAIRED_SAMPLES}, R={R_Y_UNPAIRED_SAMPLES}": model},
            X_sampler,
            Y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            experiment=experiment,
        )

        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{MAX_STEPS}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{MAX_STEPS}.pt"))

experiment.end()

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : GMMEOT_Swiss_Roll_P_XY_PAIRED_16000_Q_X_UNPAIRED_16000_R_Y_UNPAIRED_16000_LR_PAIRED_0.0003_LR_UNPAIRED_0.001_M_1_N_1_MINIBATCH_COST_rotation-v2_M_POTENTIALS_1_SHARED_HIDDEN_CHANNELS_[2]_
COMET INFO:     url                   : https://www.comet.com/muxaujl11110/light-gcot-swiss-roll/38b3c37390c84e29a5c6de7fb97907e9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     $-f(y)$ [431]                     : (6.908291621609216, 937.4696662619257)
COMET INFO:     $-f^c(x)$ [431]                   : (-944.6696971982128, -0.0018586007714693964)
COMET

Code for parameter search.

In [ ]:
import scrapbook as sb

# Choose the metric you want Optuna to minimize (e.g., your W_2 or BW_UVP calculation)
# For example, grabbing the final W_2 score:
final_mmd_score = unconditional_metrics["mmd"] # (Adjust this variable name to match your actual final metric)

# Glue it to the notebook so Optuna can read it
sb.glue("target_metric", final_mmd_score)